In [1]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data

from tqdm import tqdm

c:\Users\User\.conda\envs\signal_processing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# transaction dataset
df_tr = pd.read_csv('Data/LI-Small_Trans.csv', low_memory=False)

# account dataset
df_ac = pd.read_csv('Data/LI-Small_accounts.csv', low_memory=False)

In [4]:
df_tr['Is Laundering'].value_counts(normalize=True)*100

Is Laundering
0    99.948513
1     0.051487
Name: proportion, dtype: float64

In [4]:
df_tr.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


In [36]:
tr_size = df_tr.groupby('Account').size()

In [39]:
tr_size.describe([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

count    681281.000000
mean         10.163279
std         331.335174
min           1.000000
10%           1.000000
20%           1.000000
30%           1.000000
40%           2.000000
50%           2.000000
60%           2.000000
70%           3.000000
80%          12.000000
90%          29.000000
max      222037.000000
dtype: float64

In [25]:
df_tr[df_tr['Amount Paid'] != df_tr['Amount Received']]

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
2770,2022/09/01 00:12,394,80056EDE0,394,80056EDE0,47.610000,Euro,55.79,US Dollar,ACH,0
8081,2022/09/01 00:28,11701,800C95BF0,11701,800C95BF0,954.620000,Yuan,142.53,US Dollar,ACH,0
10451,2022/09/01 00:18,22481,80105E630,22481,80105E630,16930.030000,Yen,160.63,US Dollar,ACH,0
12948,2022/09/01 00:17,1439,8014545C0,1439,8014545C0,14.520000,UK Pound,18.76,US Dollar,ACH,0
13799,2022/09/01 00:02,20,8015D68E0,20,8015D68E0,37.000000,Euro,43.35,US Dollar,ACH,0
...,...,...,...,...,...,...,...,...,...,...,...
6924007,2022/09/10 23:57,9096,80356BD61,9096,80356BD60,0.000005,Bitcoin,0.39,Yuan,ACH,0
6924009,2022/09/10 23:30,9096,80356BD61,9096,80356BD60,0.000007,Bitcoin,0.55,Yuan,ACH,0
6924019,2022/09/10 23:38,13474,803A93631,13474,803A93630,0.000007,Bitcoin,0.08,US Dollar,ACH,0
6924021,2022/09/10 23:31,13474,803A93631,13474,803A93630,0.000020,Bitcoin,0.23,US Dollar,ACH,0


In [5]:
df_ac.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,China Bank #2820,314693,81B86A280,800D8CCF0,Corporation #41344
1,France Bank #4585,311253,8187FEA80,800B505E0,Corporation #54497
2,China Bank #2242,39996,803961E00,800D03F60,Partnership #36904
3,National Bank of Newport,331440,81B075800,801567C10,Corporation #16224
4,UK Bank #33,135417,80CF87C80,801085E00,Partnership #72930


In [6]:
df_tr.shape, df_ac.shape

((6924049, 11), (712688, 5))

In [7]:
df_tr.drop_duplicates().shape, df_ac.drop_duplicates().shape

((6924041, 11), (712688, 5))

# Edge Feature Engineering

In [ ]:
"""
GNN Feature Engineering for Anti-Money Laundering
===================================================
Builds edge features (from transactions) and initial constant node features
(from the accounts table) using pandas only.

Graph definition
----------------
  Nodes  = bank accounts  (df_ac rows)
  Edges  = transactions   (df_tr rows)

The task is *edge classification*: predict Is_Laundering per transaction.
"""


# ─────────────────────────────────────────────────────────────────────────────
# 0.  LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
df_tr = pd.read_csv("Data/LI-Small_Trans.csv", low_memory=False)
df_ac = pd.read_csv("Data/LI-Small_accounts.csv", low_memory=False)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  EDGE FEATURES  (one row per transaction)
# ─────────────────────────────────────────────────────────────────────────────

edges = df_tr.copy()

# ── 1.1  Parse timestamp ────────────────────────────────────────────────────
edges["Timestamp"] = pd.to_datetime(edges["Timestamp"])

# ── 1.2  Amount features ────────────────────────────────────────────────────
# Log-transform to compress the heavy-tailed distribution.
# We use Amount Paid (the value in payment currency).
edges["Amount_Log"] = np.log1p(edges["Amount Paid"])

# Currency exchange flag: did the currency change between sender and receiver?
# EDA showed zero laundering in cross-currency transactions in this dataset,
# but it is a useful structural signal to include.
edges["Currency_Mismatch"] = (
    edges["Receiving Currency"] != edges["Payment Currency"]
).astype(np.int8)

# ── 1.3  Temporal / cyclical features ──────────────────────────────────────
# Cyclical encoding avoids the discontinuity at midnight / end-of-week.
hour        = edges["Timestamp"].dt.hour
dow         = edges["Timestamp"].dt.dayofweek          # 0 = Monday

edges["Hour_Sin"]       = np.sin(2 * np.pi * hour / 24)
edges["Hour_Cos"]       = np.cos(2 * np.pi * hour / 24)
edges["DayOfWeek_Sin"]  = np.sin(2 * np.pi * dow  / 7)
edges["DayOfWeek_Cos"]  = np.cos(2 * np.pi * dow  / 7)

# Weekend binary flag (EDA: Sat/Sun show ~2× average laundering rate)
edges["Is_Weekend"] = (dow >= 5).astype(np.int8)

# ── 1.4  Payment format features ───────────────────────────────────────────
# ACH has ~6× the average laundering rate in this dataset → strong signal.
payment_dummies = pd.get_dummies(
    edges["Payment Format"],
    prefix="PayFmt",
    drop_first=False,       # keep all; GNN handles redundancy fine
    dtype=np.int8,
)
edges = pd.concat([edges, payment_dummies], axis=1)

# Convenience binary: ACH flag
edges["Is_ACH"] = (edges["Payment Format"] == "ACH").astype(np.int8)

# ── 1.5  Self-loop flag ─────────────────────────────────────────────────────
# Transactions where the sender and receiver account are the same.
# EDA: 11.62% of all transactions; dominated by Reinvestment (near-zero risk).
edges["Is_Self_Loop"] = (edges["Account"] == edges["Account.1"]).astype(np.int8)

# ── 1.6  Target & identifiers ───────────────────────────────────────────────
# Rename columns to standard graph edge vocabulary.
edges = edges.rename(columns={
    "Account":     "src_account",
    "Account.1":   "dst_account",
    "From Bank":   "src_bank",
    "To Bank":     "dst_bank",
    "Is Laundering": "label",
})

# ── 1.7  Final edge feature matrix ──────────────────────────────────────────
EDGE_FEAT_COLS = [
    "Amount_Log",
    "Currency_Mismatch",
    "Hour_Sin", "Hour_Cos",
    "DayOfWeek_Sin", "DayOfWeek_Cos",
    "Is_Weekend",
    "Is_ACH",
    "Is_Self_Loop",
] + list(payment_dummies.columns)

edge_features = edges[["src_account", "dst_account", "label", "Timestamp"] + EDGE_FEAT_COLS].copy()

In [12]:
print(f"Edge feature matrix: {edge_features.shape}")
edge_features[EDGE_FEAT_COLS].head()

Edge feature matrix: (6924049, 20)


,Amount_Log,Currency_Mismatch,Hour_Sin,Hour_Cos,DayOfWeek_Sin,DayOfWeek_Cos,Is_Weekend,Is_ACH,Is_Self_Loop,PayFmt_ACH,PayFmt_Bitcoin,PayFmt_Cash,PayFmt_Cheque,PayFmt_Credit Card,PayFmt_Reinvestment,PayFmt_Wire
0,14.977224,0,0.0,1.0,0.433884,-0.900969,0,0,1,0,0,0,0,0,1,0
1,7.528310,0,0.0,1.0,0.433884,-0.900969,0,0,1,0,0,0,0,0,1,0
2,13.292228,0,0.0,1.0,0.433884,-0.900969,0,0,0,0,0,0,1,0,0,0
3,2.589267,0,0.0,1.0,0.433884,-0.900969,0,0,1,0,0,0,0,0,1,0
4,7.987035,0,0.0,1.0,0.433884,-0.900969,0,0,1,0,0,0,0,0,1,0


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.  NODE FEATURES  (constant / structural — no temporal leakage)
# ─────────────────────────────────────────────────────────────────────────────
# Strategy: start from the accounts table (Bank Name, Bank ID, Entity Name).
# Constant means these are derived purely from account metadata, NOT from
# transaction statistics (which would require careful temporal masking).
# Graph-structural features (degree, etc.) are added later inside the GNN
# dataloader where the train/val/test split is respected.

nodes = df_ac.copy()

# ── 2.1  Entity type (from Entity Name prefix) ──────────────────────────────
# Entity Name looks like "Corporation #41344", "Partnership #36904", etc.
# EDA: Individual accounts have ~2× average laundering rate.
def extract_entity_type(name: str) -> str:
    for t in ("Corporation", "Individual", "Partnership", "Sole"):
        if t.lower() in str(name).lower():
            return t
    return "Other"

nodes["Entity_Type"] = nodes["Entity Name"].apply(extract_entity_type)

entity_dummies = pd.get_dummies(
    nodes["Entity_Type"],
    prefix="EntType",
    drop_first=False,
    dtype=np.int8,
)
nodes = pd.concat([nodes, entity_dummies], axis=1)

# ── 2.2  Bank identity encoding ─────────────────────────────────────────────
# Bank ID is a numeric identifier already present in the data.
# Normalise it to [0, 1] so it can be used as a raw feature.
nodes["Bank_ID_Norm"] = (
    (nodes["Bank ID"] - nodes["Bank ID"].min())
    / (nodes["Bank ID"].max() - nodes["Bank ID"].min() + 1e-9)
)

# ── 2.3  Account number as an index (NOT a feature) ─────────────────────────
# The paper explicitly excludes account IDs as features to prevent the model
# from memorising account identities rather than learning laundering patterns.
# We keep Account Number only as the node identifier for graph construction.

# ── 2.4  Final node feature matrix ──────────────────────────────────────────
NODE_FEAT_COLS = ["Bank_ID_Norm"] + list(entity_dummies.columns)

node_features = nodes[["Account Number"] + NODE_FEAT_COLS].copy()
node_features = node_features.rename(columns={"Account Number": "account_id"})

In [18]:
df_ac.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,China Bank #2820,314693,81B86A280,800D8CCF0,Corporation #41344
1,France Bank #4585,311253,8187FEA80,800B505E0,Corporation #54497
2,China Bank #2242,39996,803961E00,800D03F60,Partnership #36904
3,National Bank of Newport,331440,81B075800,801567C10,Corporation #16224
4,UK Bank #33,135417,80CF87C80,801085E00,Partnership #72930


In [17]:
print(f"\nNode feature matrix: {node_features.shape}")
node_features[NODE_FEAT_COLS].head()


Node feature matrix: (712688, 6)


,Bank_ID_Norm,EntType_Corporation,EntType_Individual,EntType_Partnership,EntType_Sole
0,0.834803,1,0,0,0
1,0.825677,1,0,0,0
2,0.106099,0,0,1,0
3,0.879228,1,0,0,0
4,0.359228,0,0,1,0


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  SAVE  (used by the GNN dataloader)
# ─────────────────────────────────────────────────────────────────────────────
edge_features.to_csv("Data/edge_features.csv", index=False)
node_features.to_csv("Data/node_features.csv", index=False)

print("\nSaved:")
print(f"  Data/edge_features.csv  — {edge_features.shape}")
print(f"  Data/node_features.csv  — {node_features.shape}")


# ─────────────────────────────────────────────────────────────────────────────
# 4.  QUICK SANITY CHECK
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Edge features (numeric stats) ──")
print(edge_features[EDGE_FEAT_COLS].describe().round(4))

print("\n── Node entity type distribution ──")
print(nodes["Entity_Type"].value_counts())

print("\n── Class balance ──")
vc = edge_features["label"].value_counts()
print(vc)
print(f"Laundering rate: {vc[1] / vc.sum() * 100:.4f}%")


Saved:
  Data/edge_features.csv  — (6924049, 20)
  Data/node_features.csv  — (712688, 6)

── Edge features (numeric stats) ──
         Amount_Log  Currency_Mismatch      Hour_Sin      Hour_Cos  \
count  6.924049e+06       6.924049e+06  6.924049e+06  6.924049e+06   
mean   7.340100e+00       1.430000e-02 -6.000000e-04  8.790000e-02   
std    3.442100e+00       1.186000e-01  6.756000e-01  7.320000e-01   
min    0.000000e+00       0.000000e+00 -1.000000e+00 -1.000000e+00   
25%    5.172600e+00       0.000000e+00 -7.071000e-01 -7.071000e-01   
50%    7.244500e+00       0.000000e+00  0.000000e+00  2.588000e-01   
75%    9.411500e+00       0.000000e+00  7.071000e-01  8.660000e-01   
max    2.892430e+01       1.000000e+00  1.000000e+00  1.000000e+00   

       DayOfWeek_Sin  DayOfWeek_Cos    Is_Weekend        Is_ACH  Is_Self_Loop  \
count   6.924049e+06   6.924049e+06  6.924049e+06  6.924049e+06  6.924049e+06   
mean    7.180000e-02  -3.936000e-01  1.226000e-01  1.150000e-01  1.162000e-01   

In [22]:
df_ac.isnull().sum()

Bank Name         0
Bank ID           0
Account Number    0
Entity ID         0
Entity Name       0
dtype: int64

In [23]:
df_ac.isnull().sum()

Bank Name         0
Bank ID           0
Account Number    0
Entity ID         0
Entity Name       0
dtype: int64

# Graph Construction

In [2]:
"""
Graph Construction for AML GNN
================================
Builds a directed temporal multigraph from the feature-engineered CSVs
and produces three graph snapshots (train / val / test) following the
exact protocol from Altman et al. (2023) — arXiv:2306.16424:

  - 60 / 20 / 20 temporal split by transaction timestamp
  - Train graph   : only training edges  (+ their endpoint nodes)
  - Val graph     : train + val edges    (evaluated on val indices only)
  - Test graph    : all edges            (evaluated on test indices only)

Each graph snapshot is a PyG HeteroData / Data object ready for GNN training.

Dependencies
------------
    pip install torch torch-geometric pandas numpy
"""


# ─────────────────────────────────────────────────────────────────────────────
# 0.  LOAD FEATURE-ENGINEERED FILES
# ─────────────────────────────────────────────────────────────────────────────
print("Loading feature files …")
edge_df = pd.read_csv("Data/edge_features.csv")
node_df = pd.read_csv("Data/node_features.csv")

# Re-parse timestamp after CSV round-trip
edge_df["Timestamp"] = pd.to_datetime(edge_df["Timestamp"])

print(f"  Edges : {len(edge_df):,}")
print(f"  Nodes : {len(node_df):,}")

Loading feature files …
  Edges : 6,924,049
  Nodes : 712,688


In [3]:
node_df.head(5)

,account_id,Bank_ID_Norm,EntType_Corporation,EntType_Individual,EntType_Partnership,EntType_Sole
0,81B86A280,0.834803,1,0,0,0
1,8187FEA80,0.825677,1,0,0,0
2,803961E00,0.106099,0,0,1,0
3,81B075800,0.879228,1,0,0,0
4,80CF87C80,0.359228,0,0,1,0


In [4]:
edge_df.head(5)

,src_account,dst_account,label,Timestamp,Amount_Log,Currency_Mismatch,Hour_Sin,Hour_Cos,DayOfWeek_Sin,DayOfWeek_Cos,Is_Weekend,Is_ACH,Is_Self_Loop,PayFmt_ACH,PayFmt_Bitcoin,PayFmt_Cash,PayFmt_Cheque,PayFmt_Credit Card,PayFmt_Reinvestment,PayFmt_Wire
0,8000ECA90,8000ECA90,0,2022-09-01 00:08:00,14.977224,0,0.0,1.0,0.433884,-0.900969,0,0,1,0,0,0,0,0,1,0
1,80021DAD0,80021DAD0,0,2022-09-01 00:21:00,7.528310,0,0.0,1.0,0.433884,-0.900969,0,0,1,0,0,0,0,0,1,0
2,8000ECA90,8006AA910,0,2022-09-01 00:00:00,13.292228,0,0.0,1.0,0.433884,-0.900969,0,0,0,0,0,0,1,0,0,0
3,8006AD080,8006AD080,0,2022-09-01 00:16:00,2.589267,0,0.0,1.0,0.433884,-0.900969,0,0,1,0,0,0,0,0,1,0
4,8006AD530,8006AD530,0,2022-09-01 00:00:00,7.987035,0,0.0,1.0,0.433884,-0.900969,0,0,1,0,0,0,0,0,1,0


In [22]:
tmp =pd.concat([edge_df["src_account"], edge_df["dst_account"]]).unique()

tmp = pd.DataFrame({
    "account_id": tmp
})

tmp_1 = tmp.merge(pd.DataFrame({'account_id': node_df['account_id'].unique()}), on="account_id", how="left", indicator=True)

In [23]:
tmp_1['_merge'].value_counts()

_merge
both          705903
left_only          0
right_only         0
Name: count, dtype: int64

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 1.  NODE INDEX MAP
#     Map every account_id string → contiguous integer index [0, N)
# ─────────────────────────────────────────────────────────────────────────────

# Union of all account IDs that appear in the accounts table OR in edges
all_accounts = pd.concat([
    node_df["account_id"],
    edge_df["src_account"],
    edge_df["dst_account"],
]).unique()

account_to_idx = {acc: idx for idx, acc in enumerate(all_accounts)}
N = len(account_to_idx)
print(f"\nTotal unique accounts (nodes): {N:,}")


# ── Node feature matrix X  [N × F_node] ──────────────────────────────────
NODE_FEAT_COLS = [c for c in node_df.columns if c != "account_id"]

# Check and remove duplicate account_ids (keep first occurrence)
n_dupes = node_df["account_id"].duplicated().sum()
if n_dupes > 0:
    print(f"  Warning: {n_dupes:,} duplicate account_id rows in node_df — keeping first")
    node_df = node_df.drop_duplicates(subset="account_id", keep="first")

# Build ordered Series: account_id → integer index
idx_series = pd.Series(account_to_idx)   # index=account_id, values=int

# Reindex node_df rows to match account_to_idx order in one shot
node_df_indexed = (
    node_df
    .set_index("account_id")
    .reindex(idx_series.index)
    [NODE_FEAT_COLS]
    .values
    .astype(np.float32)
)

X_node = torch.tensor(node_df_indexed, dtype=torch.float)
print(f"Node feature matrix : {tuple(X_node.shape)}")


Total unique accounts (nodes): 712,684
Node feature matrix : (712684, 5)


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.  TEMPORAL 60 / 20 / 20 SPLIT
#     Split is on transaction indices ordered by Timestamp — not on dates.
#     This exactly mirrors the paper's protocol.
# ─────────────────────────────────────────────────────────────────────────────

# Sort edges by timestamp (stable sort preserves original order for ties)
edge_df = edge_df.sort_values("Timestamp", kind="stable").reset_index(drop=True)

n_edges  = len(edge_df)
t1_idx   = int(n_edges * 0.60)   # end of train
t2_idx   = int(n_edges * 0.80)   # end of val

# The two cut timestamps used to build the cumulative graph snapshots
t1 = edge_df.loc[t1_idx - 1, "Timestamp"]   # last train timestamp
t2 = edge_df.loc[t2_idx - 1, "Timestamp"]   # last val timestamp

print(f"\nTemporal split")
print(f"  t1 (train end) : {t1}  →  {t1_idx:,} training edges")
print(f"  t2 (val end)   : {t2}  →  {t2_idx - t1_idx:,} validation edges")
print(f"  t_max          : {edge_df['Timestamp'].max()}  →  {n_edges - t2_idx:,} test edges")

# Boolean masks for which edges belong to each evaluation set
train_mask = edge_df.index < t1_idx
val_mask   = (edge_df.index >= t1_idx) & (edge_df.index < t2_idx)
test_mask  = edge_df.index >= t2_idx

print(f"\n  Laundering in train : {edge_df.loc[train_mask, 'label'].sum():,} | Laundering Rate in train: {edge_df.loc[train_mask, 'label'].mean() * 100:.4f}%")
print(f"  Laundering in val   : {edge_df.loc[val_mask,   'label'].sum():,}   | Laundering Rate in val: {edge_df.loc[val_mask, 'label'].mean() * 100:.4f}%")
print(f"  Laundering in test  : {edge_df.loc[test_mask,  'label'].sum():,}   | Laundering Rate in test: {edge_df.loc[test_mask, 'label'].mean() * 100:.4f}%")


Temporal split
  t1 (train end) : 2022-09-06 13:32:00  →  4,154,429 training edges
  t2 (val end)   : 2022-09-08 16:06:00  →  1,384,810 validation edges
  t_max          : 2022-09-17 15:28:00  →  1,384,810 test edges

  Laundering in train : 1,813 | Laundering Rate in train: 0.0436%
  Laundering in val   : 827   | Laundering Rate in val: 0.0597%
  Laundering in test  : 925   | Laundering Rate in test: 0.0668%


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  HELPER — build a PyG Data object from a subset of edges
# ─────────────────────────────────────────────────────────────────────────────
EDGE_FEAT_COLS = [
    c for c in edge_df.columns
    if c not in ("src_account", "dst_account", "label", "Timestamp")
]

def build_graph(edge_subset: pd.DataFrame, eval_mask: np.ndarray) -> Data:
    """
    Parameters
    ----------
    edge_subset : pd.DataFrame
        All edges that make up this graph snapshot (cumulative for val/test).
    eval_mask : np.ndarray (bool)
        Which rows of edge_subset should be evaluated (labelled).

    Returns
    -------
    PyG Data with:
        x            — node features  [N, F_node]
        edge_index   — directed edges [2, E]
        edge_attr    — edge features  [E, F_edge]
        edge_time    — Unix timestamp  [E]
        y            — labels         [E]  (−1 for non-eval edges)
        train_mask   — bool mask      [E]  (True for eval edges)
    """
    # Map accounts → integer indices
    src = edge_subset["src_account"].map(account_to_idx).values
    dst = edge_subset["dst_account"].map(account_to_idx).values
    edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)

    # Edge features
    edge_attr = torch.tensor(
        edge_subset[EDGE_FEAT_COLS].values.astype(np.float32),
        dtype=torch.float,
    )

    # Unix timestamps (seconds) — used by temporal GNN variants
    edge_time = torch.tensor(
        edge_subset["Timestamp"].astype(np.int64).values // 10**9,
        dtype=torch.long,
    )

    # Labels: −1 for context edges (not evaluated), 0/1 for eval edges
    labels = np.full(len(edge_subset), -1, dtype=np.int64)
    labels[eval_mask] = edge_subset.loc[eval_mask, "label"].values.astype(np.int64)
    y = torch.tensor(labels, dtype=torch.long)

    # Mask indicating which edges are evaluated
    eval_mask_t = torch.tensor(eval_mask, dtype=torch.bool)

    return Data(
        x          = X_node,           # shared node feature matrix
        edge_index = edge_index,
        edge_attr  = edge_attr,
        edge_time  = edge_time,
        y          = y,
        eval_mask  = eval_mask_t,
        num_nodes  = N,
    )

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.  BUILD THREE GRAPH SNAPSHOTS
# ─────────────────────────────────────────────────────────────────────────────

# ── 4a. Train graph — only training edges, evaluated on all of them ──────────
train_edges = edge_df[train_mask].reset_index(drop=True)
train_eval  = np.ones(len(train_edges), dtype=bool)   # all train edges evaluated
train_graph = build_graph(train_edges, train_eval)

# ── 4b. Val graph — train + val edges, evaluated only on val portion ─────────
val_edges_df = edge_df[train_mask | val_mask].reset_index(drop=True)
# The val edges are the last (t2_idx - t1_idx) rows of this combined frame
val_eval = np.zeros(len(val_edges_df), dtype=bool)
val_eval[t1_idx:]  = True    # rows from t1_idx onward are the val edges
val_graph = build_graph(val_edges_df, val_eval)

# ── 4c. Test graph — all edges, evaluated only on test portion ───────────────
all_edges_df = edge_df.reset_index(drop=True)   # already sorted
test_eval = np.zeros(len(all_edges_df), dtype=bool)
test_eval[t2_idx:] = True    # rows from t2_idx onward are the test edges
test_graph = build_graph(all_edges_df, test_eval)

In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.  SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
def summarise(name: str, g: Data) -> None:
    n_eval   = g.eval_mask.sum().item()
    n_laund  = (g.y[g.eval_mask] == 1).sum().item()
    laund_rt = n_laund / n_eval * 100 if n_eval > 0 else 0
    print(
        f"  {name:<12} | nodes={g.num_nodes:>7,} | "
        f"edges={g.edge_index.shape[1]:>9,} | "
        f"eval_edges={n_eval:>7,} | "
        f"laund={n_laund:>5,} ({laund_rt:.4f}%)"
    )

print("\n── Graph snapshots ──────────────────────────────────────────────────────")
print(f"  {'Name':<12} | {'nodes':>8} | {'edges':>10} | {'eval_edges':>10} | laundering")
print(f"  {'-'*70}")
summarise("train_graph", train_graph)
summarise("val_graph",   val_graph)
summarise("test_graph",  test_graph)

print(f"\nEdge feature dim : {train_graph.edge_attr.shape[1]}")
print(f"Node feature dim : {train_graph.x.shape[1]}")


── Graph snapshots ──────────────────────────────────────────────────────
  Name         |    nodes |      edges | eval_edges | laundering
  ----------------------------------------------------------------------
  train_graph  | nodes=712,684 | edges=4,154,429 | eval_edges=4,154,429 | laund=1,813 (0.0436%)
  val_graph    | nodes=712,684 | edges=5,539,239 | eval_edges=1,384,810 | laund=  827 (0.0597%)
  test_graph   | nodes=712,684 | edges=6,924,049 | eval_edges=1,384,810 | laund=  925 (0.0668%)

Edge feature dim : 16
Node feature dim : 5


In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.  SAVE
# ─────────────────────────────────────────────────────────────────────────────
torch.save(train_graph, "Data/train_graph.pt")
torch.save(val_graph,   "Data/val_graph.pt")
torch.save(test_graph,  "Data/test_graph.pt")

print("\nSaved:")
print("  Data/train_graph.pt")
print("  Data/val_graph.pt")
print("  Data/test_graph.pt")

# ── Reload check ─────────────────────────────────────────────────────────────
g = torch.load("Data/train_graph.pt", weights_only=False)
print(f"\nReload check → train_graph edges: {g.edge_index.shape[1]:,}  ✓")


Saved:
  Data/train_graph.pt
  Data/val_graph.pt
  Data/test_graph.pt

Reload check → train_graph edges: 4,154,429  ✓
